# Script 05 Local Test: CK_skin_res

Run the simplified script `05_apply_qc_filters_for_reclustering.py` locally on the small VBCT `CK_skin_res` sample. This notebook tests the real script and then inspects the filtered output object and compact summary CSV.

In [ ]:
from pathlib import Path
import json

config_path = Path("config/05_apply_qc_filters/vbct/CK_skin_res.json")
config_path


In [ ]:
with config_path.open() as f:
    cfg = json.load(f)

cfg


In [ ]:
project = cfg["project"]
dataset_name = cfg["dataset_name"]
output_label = cfg.get("output_label", "filtered_qc_v1")
base_dir = Path(cfg.get("base_dir", "data/xenium"))

processed_path = base_dir / "processed" / project / dataset_name
input_h5ad = Path(cfg.get("input_h5ad", processed_path / f"adata_expression_clean_{dataset_name}_qc_annotated.h5ad"))
filtered_output_h5ad = Path(cfg.get("filtered_output_h5ad", processed_path / f"adata_expression_clean_{dataset_name}_qc_{output_label}.h5ad"))
annotated_output_h5ad = Path(cfg.get("annotated_output_h5ad", processed_path / f"adata_expression_clean_{dataset_name}_qc_annotated_{output_label}.h5ad"))
summary_csv = base_dir / "output" / project / "QC_filtering" / dataset_name / f"{dataset_name}_qc_filter_summary_{output_label}.csv"

{
    "input_h5ad": input_h5ad,
    "annotated_output_h5ad": annotated_output_h5ad,
    "filtered_output_h5ad": filtered_output_h5ad,
    "summary_csv": summary_csv,
}


In [ ]:
if not input_h5ad.exists():
    raise FileNotFoundError(
        "Missing script 01 QC-annotated AnnData input. Run script 01 for CK_skin_res first: "
        f"{input_h5ad}"
    )

input_h5ad


In [ ]:
%run 05_apply_qc_filters_for_reclustering.py --config config/05_apply_qc_filters/vbct/CK_skin_res.json


In [ ]:
import pandas as pd

summary = pd.read_csv(summary_csv)
summary


In [ ]:
import anndata as ad

adata_annotated = ad.read_h5ad(annotated_output_h5ad)
adata_filtered = ad.read_h5ad(filtered_output_h5ad)

{
    "annotated_cells": adata_annotated.n_obs,
    "filtered_cells": adata_filtered.n_obs,
    "removed_cells": adata_annotated.n_obs - adata_filtered.n_obs,
    "filtered_genes": adata_filtered.n_vars,
}


In [ ]:
adata_annotated.obs["qc_filter_status"].value_counts(dropna=False)


In [ ]:
adata_annotated.obs[[
    "min_trans_passed",
    "max_trans_threshold_passed",
    "negative_control_probe_ge2",
    "max_area_threshold_99_by_cluster",
    "qc_keep_for_reclustering",
    "qc_filter_status",
]].head()
